# Chapter-15: Fundamentals Of LangChain Agents

# <font color="#800000">Fundamentals Of LangChain Agents</font>

## <font color="#3366FF">Fundamentals Of LangChain Agent</font>

### Basic Example Of LangChain Agent

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.agents import create_agent

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Define Agent
agent = create_agent(
    model=llm
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What is black hole. Explain in one line.")]})

## Print Messages
response['messages'][-1].pretty_print()


================================== Ai Message ==================================

A black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape from within its event horizon.


#### Add System Prompt

In [3]:
## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant. Respond in one line."
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Tell a fact about universe.")]})

## Print Messages
response['messages'][-1].pretty_print()


================================== Ai Message ==================================

The observable universe is about 93 billion light-years in diameter.


### Example of Agent With Tool

In [4]:
import os
import random
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    price = round(random.uniform(10, 500), 2)
    return price

## Define Agent
agent = create_agent(
    model=llm,
    tools=[stock_price],
    system_prompt="You are a helpful stock market assistant",
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What is stock price of Google")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

What is stock price of Google
================================== Ai Message ==================================
Tool Calls:
  stock_price (call_q7XAnN6iHS2AH8I0a914drkt)
 Call ID: call_q7XAnN6iHS2AH8I0a914drkt
  Args:
    name: Alphabet Inc. (GOOGL)
================================= Tool Message =================================
Name: stock_price

165.19
================================== Ai Message ==================================

Alphabet Inc. (ticker GOOGL) is trading at $165.19 per share (USD).  

Would you like the price for GOOG (Class C), the intraday change/percent, or recent historical data?


## <font color="#3366FF">Implementing Short-Term Memory</font>

### Example Of Short-Term Memory

In [5]:
import os
import random
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    price = round(random.uniform(10, 500), 2)
    return price

## Define Agent
agent = create_agent(
    model=llm,
    tools=[stock_price],
    system_prompt="You are a helpful assistant",
    checkpointer=InMemorySaver(),  
)

## Create config with thread ID
config = {"configurable": {"thread_id": "s1"}}

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What is stock price of Nvidia?")]},config)

## Print Messages
response['messages'][-1].pretty_print()


================================== Ai Message ==================================

NVIDIA (NVDA) current market price: $367.94 (USD).

Prices change constantly — would you like the intraday percent change, a recent chart, or historical prices?


#### Asking Follow-up Question

In [6]:
## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What this company does, tell in one line")]},config)

## Print Messages
response['messages'][-1].pretty_print()


================================== Ai Message ==================================

NVIDIA designs and manufactures graphics processing units (GPUs) and AI computing platforms used for gaming, data centers, professional visualization, and autonomous systems.


## <font color="#3366FF">Context Schema</font>

### Example Of Context Schema

In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Custom Context Schema
@dataclass
class TravelContext:
    travel_location_preference: str
    travel_type_preference: str = "beach"


## -------- Tools for getting context data --------
@tool
def get_travel_location_preference(runtime: ToolRuntime) -> str:
    """Get the user's preferred travel location"""
    return runtime.context.travel_location_preference
    
@tool
def get_travel_type_preference(runtime: ToolRuntime) -> str:
    """Get the user's preferred travel type"""
    return runtime.context.travel_type_preference

## Define Agent
agent = create_agent(
    model=llm,
    tools=[get_travel_type_preference, get_travel_location_preference],
    system_prompt="You are a helpful Travel Assistant. Respond in one line.",
    context_schema=TravelContext  
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Please suggest a holiday place?")]},                      context=TravelContext(travel_location_preference="India") )

## Print Messages
response['messages'][-1].pretty_print()


/Users/ankur/Desktop/Langchain_Review/.venv/lib/python3.13/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=TravelContext(travel_loca...type_preference='beach'), input_type=TravelContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/ankur/Desktop/Langchain_Review/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=TravelContext(travel_loca...type_preference='beach'), input_type=TravelContext])
  return self.__pydantic_serializer__.to_python(


================================== Ai Message ==================================

I suggest Radhanagar Beach on Havelock Island (Andaman Islands) — pristine white sand, crystal-clear water, great snorkeling, and a peaceful tropical escape.


In [8]:
## Print All Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

Please suggest a holiday place?
================================== Ai Message ==================================
Tool Calls:
  get_travel_type_preference (call_vjc93OIzXMlXC0zHZTacteQP)
 Call ID: call_vjc93OIzXMlXC0zHZTacteQP
  Args:
  get_travel_location_preference (call_8SEQRgYhhETRFOCKunht5zUe)
 Call ID: call_8SEQRgYhhETRFOCKunht5zUe
  Args:
================================= Tool Message =================================
Name: get_travel_type_preference

beach
================================= Tool Message =================================
Name: get_travel_location_preference

India
================================== Ai Message ==================================

I suggest Radhanagar Beach on Havelock Island (Andaman Islands) — pristine white sand, crystal-clear water, great snorkeling, and a peaceful tropical escape.


## <font color="#3366FF">State Schema</font>

### Example Of State Schema

In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Custom State Schema
class TravelState(AgentState):
    travel_location_preference: str
    travel_type_preference: str


## -------- Tools for updating custom states --------
@tool
def update_travel_location_preference(travel_location_preference: str, runtime: ToolRuntime) -> Command:
    """Update user's preferred travel location (like country, state, city, etc.) when they reveal."""
    return Command(update={
        "travel_location_preference": travel_location_preference, 
        "messages": [ToolMessage("Successfully updated travel location preferrence", tool_call_id=runtime.tool_call_id)]}
        )
@tool
def update_travel_type_preference(travel_type_preference: str, runtime: ToolRuntime) -> Command:
    """Update user's preferred travel type when they reveal."""
    return Command(update={
        "travel_type_preference": travel_type_preference, 
        "messages": [ToolMessage("Successfully updated travel type preference", tool_call_id=runtime.tool_call_id)]}
        )


## -------- Tools for getting custom state values --------

@tool
def get_travel_location_preference(runtime: ToolRuntime) -> str:
    """Read user's preferred travel location from the state."""
    try:
        return runtime.state["travel_location_preference"]
    except KeyError:
        return "User's preferred travel location not found in state"

@tool
def get_travel_type_preference(runtime: ToolRuntime) -> str:
    """Read user's preferred travel type from the state."""
    try:
        return runtime.state["travel_type_preference"]
    except KeyError:
        return "User's preferred travel type not found in state"
    

## Define Agent
agent = create_agent(
    model=llm,
    tools=[update_travel_location_preference, update_travel_type_preference,get_travel_location_preference, get_travel_type_preference],
    system_prompt="You are a helpful Travel Assistant. Respond in one line.",
    checkpointer=InMemorySaver(),
    state_schema=TravelState
)


#### Invoke Agent

In [10]:
## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Please suggest a holiday plan preferably in India?")]},
                       {"configurable": {"thread_id": "1"}})

## Print Messages
response['messages'][-1].pretty_print()

## Print user's preferred travel location
try:
    print("\ntravel_location_preference:", response["travel_location_preference"])   
except KeyError:
    print("\ntravel_location_preference: Not Available")  

## Print user's preferred travel type
try:
    print("\ntravel_type_preference:", response["travel_type_preference"])   
except KeyError:
    print("\ntravel_type_preference: Not Available")


================================== Ai Message ==================================

7-day India holiday: Days 1–2 Delhi (Red Fort, Qutub Minar, Chandni Chowk), Day 3 Agra (Taj Mahal, Agra Fort, optional sunrise), Days 4–7 Jaipur (Amber Fort, City Palace, Hawa Mahal; optional Ranthambore day-safari or push to 10 days for Udaipur/Jodhpur)—want this heritage loop or a beach/mountain/adventure plan instead?

travel_location_preference: India

travel_type_preference: Not Available


#### Follow-up Conversation - 1

In [11]:
## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Am interested in beaches")]},
                       {"configurable": {"thread_id": "1"}})

## Print Messages
response['messages'][-1].pretty_print()

## Print user's preferred travel location
try:
    print("\ntravel_location_preference:", response["travel_location_preference"])   
except KeyError:
    print("\ntravel_location_preference: Not Available")  

## Print user's preferred travel type
try:
    print("\ntravel_type_preference:", response["travel_type_preference"])   
except KeyError:
    print("\ntravel_type_preference: Not Available")


================================== Ai Message ==================================

7-day beach holiday: Days 1–3 Goa (north beaches/Calangute for watersports + south/Palolem for relaxed vibes, sunset cruise, nightlife), Day 4 travel to Gokarna (Om/Kudle for chilled beach time), Days 5–7 Kerala (Varkala cliffs for yoga and sunsets or Marari for quiet beach + optional Ayurvedic massage) — or choose a 5–7 day Andaman alternative (Port Blair + Havelock’s Radhanagar Beach); which do you prefer?

travel_location_preference: India

travel_type_preference: beach


#### Follow-up Conversation - 2

In [13]:
## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Can you give options for Thailand")]},
                       {"configurable": {"thread_id": "1"}})

## Print Messages
response['messages'][-1].pretty_print()

## Print user's preferred travel location
try:
    print("\ntravel_location_preference:", response["travel_location_preference"])   
except KeyError:
    print("\ntravel_location_preference: Not Available")  

## Print user's preferred travel type
try:
    print("\ntravel_type_preference:", response["travel_type_preference"])   
except KeyError:
    print("\ntravel_type_preference: Not Available")


================================== Ai Message ==================================

5–7 day Thailand beach options: 1) Phuket + nearby islands (Patong, Kata, Phi Phi day trip, Phang Nga Bay), 2) Krabi (Ao Nang, Railay Beach, island-hopping to Koh Hong), 3) Koh Samui + Koh Phangan (relaxation + full-moon/half-moon parties), 4) Koh Tao (diving-focused), 5) Trang/Trang Islands (less crowded pristine beaches) — want a detailed 7-day itinerary for any of these?

travel_location_preference: Thailand

travel_type_preference: beach


## <font color="#3366FF">Structured Output</font>

### Example Of Provider Strategy

In [14]:
import os
from dotenv import load_dotenv
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Custom Output Schema
class OutputSchema(TypedDict):
    fact: str 
    joke: str 

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    response_format=ProviderStrategy(OutputSchema)
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Write one line fact and joke about mouse")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

Write one line fact and joke about mouse
================================== Ai Message ==================================

{"fact":"A mouse's incisors grow continuously throughout its life, so it must gnaw constantly to keep them worn down.","joke":"Why did the mouse cross the keyboard? To get to the other website."}


### Example Of Tool Strategy

In [15]:
import os
from dotenv import load_dotenv
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Custom Output Schema
class Profile(TypedDict):
    name: str 
    location: str 
    hobbies: list[str]

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You extract profile details",
    response_format=ToolStrategy(Profile)
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="John lives in New York and loves dancing and trekking")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

John lives in New York and loves dancing and trekking
================================== Ai Message ==================================
Tool Calls:
  Profile (call_XMHe4QWALCuBQ1fwjnBSu7qS)
 Call ID: call_XMHe4QWALCuBQ1fwjnBSu7qS
  Args:
    name: John
    location: New York
    hobbies: ['dancing', 'trekking']
================================= Tool Message =================================
Name: Profile

Returning structured response: {'name': 'John', 'location': 'New York', 'hobbies': ['dancing', 'trekking']}


## <font color="#3366FF">Streaming</font>

### Example Of Streaming Messages And Updates

In [16]:
import os
import random
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    price = round(random.uniform(10, 500), 2)
    return price

## Define Agent
agent = create_agent(
    model=llm,
    tools=[stock_price],
    system_prompt="You are a helpful stock market assistant",
)


#### Streaming In Messages Mode

In [17]:
## Stream agent
for token, metadata in agent.stream({"messages": [HumanMessage(content="What is stock price of Microsoft and NVIDIA")]}, stream_mode="messages"):
    if token.content:  # Check if there's actual content
        if metadata['langgraph_node'] != 'tools': # Make sure its not tools call content
            print(token.content, end="", flush=True)  # Print token


Here are the current share prices:

- Microsoft (MSFT): $83.63
- NVIDIA (NVDA): $163.24

Prices are per share (USD) and can change rapidly — tell me if you want percent change, intraday chart, or recent news.

#### Streaming In Updates Mode

In [18]:
## Stream agent
for chunk in agent.stream({"messages": [HumanMessage(content="What is stock price of Google")]}, stream_mode="updates"):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")


step: model
content: [{'type': 'tool_call', 'name': 'stock_price', 'args': {'name': 'GOOGL'}, 'id': 'call_wrQ60n5psLdYpnxEyHm9wPpD'}]
step: tools
content: [{'type': 'text', 'text': '446.94'}]
step: model
content: [{'type': 'text', 'text': 'Alphabet Inc. (Google) — ticker GOOGL — is currently $446.94 per share (USD).\n\nNote: this price may be delayed. Do you want the price for GOOG (Class C) as well, or additional details (day change, intraday chart, market timestamp)?'}]


### Example Of Streaming Custom Data

In [19]:
import os
import random
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.config import get_stream_writer  


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    
    ## Initialize Stream Writer
    writer = get_stream_writer() 

    ## Stream custom data
    writer(f"Connecting to stock exchange server..")
    writer(f"Retrieving stock price of : {name}..")
    
    price = round(random.uniform(10, 500), 2)
    writer(f"Returning Stock price of {name}.")

    return price

## Define Agent
agent = create_agent(
    model=llm,
    tools=[stock_price],
    system_prompt="You are a helpful stock market assistant",
)

## Stream in custom mode
for chunk in agent.stream({"messages": [HumanMessage(content="What is stock price of Google")]},
    stream_mode="custom"
):
    print(chunk)


Connecting to stock exchange server..
Retrieving stock price of : GOOGL..
Returning Stock price of GOOGL.
Connecting to stock exchange server..
Retrieving stock price of : GOOG..
Returning Stock price of GOOG.


### Example Of Streaming In Multiple Modes

In [20]:
## Stream agent in both updates and custom modes.
for stream_mode, chunk in agent.stream(  
    {"messages": [HumanMessage(content="What is stock price of Google")]},
    stream_mode=["updates", "custom"]
):
    print(f"stream_mode: {stream_mode}")
    if stream_mode == 'updates':
        for step, data in chunk.items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")
    else:
        print(f"content: {chunk}")
    print("\n")


stream_mode: updates
step: model
content: [{'type': 'tool_call', 'name': 'stock_price', 'args': {'name': 'GOOGL'}, 'id': 'call_4CqK4qIIgMC5dKOZl6rfjgxq'}, {'type': 'tool_call', 'name': 'stock_price', 'args': {'name': 'GOOG'}, 'id': 'call_0RX7XjDGsPD1IiWvzvsgqgyA'}]


stream_mode: custom
content: Connecting to stock exchange server..


stream_mode: custom
content: Retrieving stock price of : GOOG..


stream_mode: custom
content: Returning Stock price of GOOG.


stream_mode: custom
content: Connecting to stock exchange server..


stream_mode: custom
content: Retrieving stock price of : GOOGL..


stream_mode: custom
content: Returning Stock price of GOOGL.


stream_mode: updates
step: tools
content: [{'type': 'text', 'text': '459.33'}]


stream_mode: updates
step: tools
content: [{'type': 'text', 'text': '29.45'}]


stream_mode: updates
step: model
content: [{'type': 'text', 'text': 'Do you mean Alphabet Inc. (Google)? It trades under two tickers:\n\n- GOOGL (Class A): $29.45 per share  \